# الفصل 41: أسس القياس الكمّي

> قبل أي حديث عن Benchmarks معقّدة، يجب أن تفهم بعمق المقاييس الأساسية ولماذا تخدعك في كثير من الأحيان.

- **الجزء:** 9 — التقييم الشامل
- **الفصل:** 41
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 40: النماذج الموحّدة: Any-to-Any
- **الوسوم:** Perplexity، BLEU، ROUGE، BERTScore

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch, transformers
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch"),
    ("transformers", "transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch41-evaluation-metrics.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل أي حديث عن Benchmarks معقّدة، يجب أن تفهم بعمق المقاييس الأساسية ولماذا تخدعك في كثير من الأحيان.

> **بيانات الفصل**
>
> - **الجزء:** 9 — التقييم الشامل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 40: النماذج الموحّدة: Any-to-Any


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Perplexity — ماذا تقيس فعليًا؟
- فهم BLEU وROUGE — مقاييس التداخل اللفظي (Overlap-based)
- فهم BERTScore — خطوة نحو الدلالة


## 1. لماذا يهمّك هذا الموضوع؟

قبل أي حديث عن Benchmarks معقّدة، يجب أن تفهم بعمق المقاييس الأساسية ولماذا **تخدعك** في كثير من الأحيان. أكبر خطأ يرتكبه المهندسون الجدد هو الثقة العمياء برقم واحد (مثل Perplexity أو BLEU) دون فهم ما يقيسه فعليًا وما لا يقيسه.


## 2. الخريطة الذهنية


```text
تقييم LLM
    │
    ├── مقاييس تلقائية إحصائية (سريعة، رخيصة، محدودة الدلالة)
    │   ├── Perplexity
    │   ├── BLEU / ROUGE / METEOR
    │   └── BERTScore
    │
    ├── مقاييس قائمة على المهمة (Task-Specific Accuracy)
    │
    ├── تقييم بواسطة نموذج آخر (LLM-as-a-Judge)
    │
    └── تقييم بشري (Human Evaluation) — المرجع الذهبي لكن الأبطأ والأغلى
```


## 3. الشرح المتدرج

### 🟢 أ) Perplexity — ماذا تقيس فعليًا؟

راجعنا Perplexity في الفصل الأول كمقياس تدريب، لكن كمقياس **تقييم نهائي** لها قيود جوهرية:

$$
\text{PPL}(x) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(x_i \mid x_{<i})\right)
$$

> ⚠️ **خطأ شائع جدًا**: مقارنة Perplexity بين نموذجين يستخدمان **tokenizers مختلفة**. إذا كان tokenizer أ يُفتّت النص لرموز أقل (fertility أقل، راجع الفصل 19)، فإن PPL لديه ستكون أعلى تلقائيًا بغضّ النظر عن جودة النموذج الفعلية — المقارنة غير عادلة إطلاقًا إلا بتوحيد وحدة القياس (bits-per-byte بدلاً من bits-per-token).


In [ ]:
import torch
import torch.nn.functional as F

def compute_bits_per_byte(model, tokenizer, text: str) -> float:
    """
    مقياس عادل للمقارنة عبر tokenizers مختلفة:
    نحسب log-likelihood الكلي ونقسمه على عدد البايتات الخام للنص
    (وليس عدد الرموز، الذي يختلف حسب tokenizer)
    """
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        # outputs.loss هو متوسط NLL لكل رمز (log base e)
        total_nll_nats = outputs.loss.item() * (input_ids.shape[1] - 1)

    num_bytes = len(text.encode("utf-8"))
    bits_per_byte = (total_nll_nats / 0.6931) / num_bytes   # تحويل nats -> bits (log2)
    return bits_per_byte


> 💡 **نصيحة عملية**: عند تقرير نتائج بحثية أو مقارنة نماذج بـ tokenizers مختلفة (مثل مقارنة نموذج عربي مخصص بنموذج LLaMA متعدد اللغات)، استخدم **دائمًا** `bits-per-byte` كوحدة موحّدة، لا `perplexity` الخام.

### 🟡 ب) BLEU وROUGE — مقاييس التداخل اللفظي (Overlap-based)

**BLEU** (للترجمة الآلية أساسًا) يقيس تطابق n-grams بين النص المُولَّد والمرجع:

$$
\text{BLEU} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)
$$

حيث $p_n$ دقة n-gram، و $\text{BP}$ عقوبة الطول القصير (Brevity Penalty):

$$
\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{(1 - r/c)} & \text{if } c \le r \end{cases}
$$


In [ ]:
from collections import Counter
import math

def compute_bleu(candidate: list, reference: list, max_n=4) -> float:
    """تطبيق مبسّط وتعليمي لـ BLEU من الصفر"""
    def get_ngrams(tokens, n):
        return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)])

    precisions = []
    for n in range(1, max_n + 1):
        cand_ngrams = get_ngrams(candidate, n)
        ref_ngrams = get_ngrams(reference, n)
        overlap = sum(min(count, ref_ngrams[ng]) for ng, count in cand_ngrams.items())
        total = max(sum(cand_ngrams.values()), 1)
        precisions.append(overlap / total if total > 0 else 0)

    if min(precisions) == 0:
        return 0.0

    geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if len(candidate) > len(reference) else math.exp(1 - len(reference) / max(len(candidate), 1))
    return bp * geo_mean

def compute_rouge_l(candidate: list, reference: list) -> dict:
    """ROUGE-L: يعتمد على أطول تسلسل فرعي مشترك (LCS) — أكثر ملاءمة للتلخيص من BLEU"""
    def lcs_length(a, b):
        dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
        for i in range(1, len(a) + 1):
            for j in range(1, len(b) + 1):
                if a[i-1] == b[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        return dp[-1][-1]

    lcs = lcs_length(candidate, reference)
    precision = lcs / len(candidate) if candidate else 0
    recall = lcs / len(reference) if reference else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1}


> ⚠️ **المشكلة الجوهرية في BLEU/ROUGE**: تعتمد على **التطابق اللفظي الحرفي**. جملتان بنفس المعنى تمامًا لكن بصياغة مختلفة (ترادف) تحصلان على درجة منخفضة. هذا قاتل بشكل خاص للغة العربية الغنية بالترادف والتصريف.

### 🟡 ج) BERTScore — خطوة نحو الدلالة

بدلاً من التطابق الحرفي، BERTScore يقيس التشابه في **فضاء embeddings السياقية**:

$$
\text{BERTScore}_{\text{Recall}} = \frac{1}{|x|}\sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} \cos(\mathbf{x}_i, \hat{\mathbf{x}}_j)
$$


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

def compute_bertscore(candidate: str, reference: str, model_name="bert-base-multilingual-cased"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    def get_embeddings(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
        return F.normalize(outputs.last_hidden_state[0], dim=-1)  # [seq_len, hidden]

    cand_embeds = get_embeddings(candidate)
    ref_embeds = get_embeddings(reference)

    similarity_matrix = cand_embeds @ ref_embeds.T   # [cand_len, ref_len]
    recall = similarity_matrix.max(dim=0).values.mean().item()
    precision = similarity_matrix.max(dim=1).values.mean().item()
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1}


## 4. مثال محلول خطوة بخطوة

لنحسب BLEU-2 مبسّطًا. المرجع: `القطة تجلس على السجادة`. التوليد: `القطة تجلس فوق السجادة`.

**Unigrams المولّدة:** القطة، تجلس، فوق، السجادة (4).
- من أصل 4، ثلاث كلمات موجودة في المرجع (القطة، تجلس، السجادة) → precision-1 = 3/4 = 0.75.

**Bigrams المولّدة:** (القطة، تجلس)، (تجلس، فوق)، (فوق، السجادة) (3).
- (القطة، تجلس) موجودة، (تجلس، فوق) غير موجودة، (فوق، السجادة) غير موجودة → precision-2 = 1/3 ≈ 0.333.

**المتوسط الهندسي:** $\sqrt{0.75 \times 0.333} \approx 0.5$.

**Brevity penalty:** طول التوليد 4، المرجع 4 → BP = 1. النتيجة النهائية ≈ 0.5.

**ماذا يعني هذا؟** الكلمة `فوق` بدل `على` خفضت BLEU كثيرًا رغم أن المعنى واحد. هذا يوضح لماذا BLEU وحده ظالم، ولماذا نحتاج BERTScore أو حكمًا بشريًا.


## 5. نصائح وحيل

> 💡 **أبلغ دائمًا عن فترة ثقة**: متوسط 3 نقاط على 100 مثال قد يكون غير دال إحصائيًا. استخدم bootstrap أو t-test.

> 💡 **طبّع النص العربي قبل BLEU/ROUGE**: وحّد الألف (أ/إ/آ → ا)، التاء المربوطة، التشكيل، والمسافات. بدون ذلك تقيس الإملاء لا الجودة.

> 💡 **لا تقارن perplexity بين tokenizers مختلفة**: نموذج بـ32k vocab وآخر بـ128k ليسا على نفس المقياس. استخدم bits-per-character.

> 💡 **استخدم pass@k للبرمجة، لا exact match**: نموذج قد ينجح في المحاولة الثالثة؛ هذا نجاح حقيقي.

> 💡 **أضف عيّنة بشرية صغيرة**: 30-50 حكمًا بشريًا تكشف ما لا تكشفه المقاييس.


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاعتماد على مقياس واحد**: BLEU مرتفع لا يعني إجابة صحيحة أو آمنة. اجمع بين مقياس إحصائي + دلالي + بشري.

> ⚠️ **مقارنة perplexity عبر نماذج/tokenizers مختلفة**: الأرقام غير قابلة للمقارنة مباشرة. قارن على نفس التجزئة أو استخدم BPC.

> ⚠️ **استخدام exact match لمهام توليدية**: صياغة مختلفة صحيحة ستُحسب خطأ. استخدم LLM-judge أو BERTScore.

> ⚠️ **إهمال التباين**: تقييم على 20 مثالًا يعطي نتائج عشوائية. استخدم 200+ مثال وفترة ثقة.

> ⚠️ **تجاهل التطبيع العربي**: بدون تطبيع، اختلاف الهمزات والتشكيل يُخفّض الدرجات زورًا.


## 7. جدول مقارنة

| المقياس | يقيس | الحساسية للترادف | التكلفة | الاستخدام المناسب |
|---|---|---|---|---|
| **Perplexity** | جودة التوزيع الاحتمالي | غير قابل للتطبيق هنا | زهيدة جدًا | مقارنة نماذج بنفس tokenizer فقط |
| **BLEU** | تطابق n-grams (دقّة) | منخفضة جدًا | زهيدة | ترجمة آلية (محدود الفائدة اليوم) |
| **ROUGE-L** | أطول تسلسل مشترك | منخفضة | زهيدة | التلخيص |
| **BERTScore** | تشابه دلالي بالـ embeddings | عالية | متوسطة | أي مهمة توليد نصي عام |
| **LLM-as-a-Judge** | جودة شاملة متعددة الأبعاد | عالية جدًا | عالية | التقييم النهائي الشامل |

✅ **أفضل الممارسات**: لا تعتمد **أبدًا** على مقياس واحد. استخدم لوحة قياس (dashboard) متعددة الأبعاد: مقياس إحصائي سريع للمراقبة أثناء التطوير + LLM-as-a-Judge للتقييم الدوري + تقييم بشري دوري على عيّنة صغيرة كـ "نقطة حقيقة" (ground truth check).

---


## 8. ملخص أفضل الممارسات

- ✅ استخدم لوحة مقاييس متعددة: إحصائي سريع + دلالي + حكم بشري دوري.
- ✅ طبّع النص العربي ووثّق خطوات التطبيع.
- ✅ أبلغ عن عدد العينات وفترة الثقة.
- ✅ لا تقارن perplexity بين tokenizers مختلفة.
- ✅ اختر المقياس حسب المهمة: BLEU للترجمة، ROUGE للتلخيص، pass@k للبرمجة، ECE للثقة.


## 9. أسئلة مقابلات

**س1: لماذا قد يحصل نموذجان على نفس BLEU بجودة مختلفة؟**

**الإجابة النموذجية:** BLEU يقيس تداخل n-grams مع مرجع واحد، متجاهلًا المعنى والطلاقة والتماسك. نموذج قد يستخدم مرادفات صحيحة فينخفض BLEU، وآخر يكرّر عبارات المرجع دون فهم فيرتفع. لذلك BLEU مؤشر سطحي، ويجب دمجه مع BERTScore وحكم بشري.

**س2: ما الفرق بين perplexity وbits-per-character؟**

**الإجابة النموذجية:** Perplexity = exp(متوسط cross-entropy لكل token)، وتعتمد على tokenizer. BPC يقيس عدد البتات لكل حرف، فيسمح بمقارنة نماذج بتجزئات مختلفة. إذا أردت مقارنة عادلة عبر tokenizers، استخدم BPC.

**س3: متى يكون pass@k أنسب من exact match؟**

**الإجابة النموذجية:** في البرمجة والمسائل متعددة الحلول، لأن هناك عدة برامج صحيحة. pass@k يقيس احتمال أن تكون واحدة من k محاولات صحيحة، وهو أقرب لقدرة النموذج العملية. exact match يصلح فقط عندما تكون الإجابة فريدة وقصيرة.


## 10. تمرين عملي

1. خذ 5 أزواج (مرجع، إعادة صياغة بمرادفات) بالعربية، واحسب BLEU وBERTScore.
2. طبّق تطبيعًا عربيًا (همزات، تشكيل) وأعد الحساب؛ لاحظ الفرق.
3. احسب perplexity لنموذجين صغيرين على نفس النص، ثم احسب BPC لكل منهما.
4. صمّم مجموعة 50 سؤالًا مع إجابات مرجعية، وحدّد أي مقياس ستستخدم لكل نوع.
5. اكتب فقرة عن حدود المقاييس الإحصائية في تقييم العربية.


## 11. مراجع للتعمّق

- Papineni, K., et al. (2002). *BLEU: a Method for Automatic Evaluation of Machine Translation*. ACL.
- Lin, C.-Y. (2004). *ROUGE: A Package for Automatic Evaluation of Summaries*. ACL Workshop.
- Zhang, T., et al. (2020). *BERTScore: Evaluating Text Generation with BERT*. ICLR.
- Chen, M., et al. (2021). *Evaluating Large Language Models Trained on Code* (pass@k).
- Guo, C., et al. (2017). *On Calibration of Modern Neural Networks*. ICML.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التقييم الشامل / الفصل 41: أسس القياس الكمّي.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
